In [6]:
#import the necessary libraries
import pandas as pd
import numpy as np
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt


In [7]:
#load the dataset
df = pd.read_csv("Sydney_Housing_Data.csv")

print("Total properties:", len(df))
print(df['Suburb'].value_counts())

Total properties: 102
Suburb
Mount Druitt    36
Parramatta      36
Campbelltown    30
Name: count, dtype: int64


In [8]:
#summary of the dataset
summary = df.groupby('Suburb')['Sale_Price_AUD'].agg(['count', 'median', 'min', 'max'])
print(summary)

missing_land = df.groupby('Suburb')['Land_Size_sqm'].apply(lambda x: round(x.isna().mean() * 100, 1))
print(missing_land)

              count     median       min        max
Suburb                                             
Campbelltown     30   958000.0  620000.0  1750000.0
Mount Druitt     36  1021250.0  440000.0  1700000.0
Parramatta       36   589000.0  398000.0   800000.0
Suburb
Campbelltown    20.0
Mount Druitt    25.0
Parramatta      86.1
Name: Land_Size_sqm, dtype: float64


In [18]:
df = pd.read_csv("Sydney_Housing_Data.csv")
df["Sale_Date"] = pd.to_datetime(df["Sale_Date"])

#price distribution and suburb comparison
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
order = ["Mount Druitt", "Campbelltown", "Parramatta"]
axes[0].boxplot([df[df.Suburb == s]["Sale_Price_AUD"] for s in order], tick_labels=order)
axes[0].set_title("Sale Price Distribution by Suburb")
axes[0].set_ylabel("Sale Price (AUD)")
for s in order:
    axes[1].hist(df[df.Suburb == s]["Sale_Price_AUD"], bins=12, alpha=0.5, label=s)
axes[1].set_title("Sale Price Histogram by Suburb")
axes[1].legend()
plt.tight_layout()
plt.savefig("price_distribution.png", dpi=120)


In [ ]:
#outlier detection
print("Outliers")
for s in order:
    sub = df[df.Suburb == s]
    q1, q3 = sub.Sale_Price_AUD.quantile([0.25, 0.75])
    iqr = q3 - q1
    flagged = sub[(sub.Sale_Price_AUD < q1 - 1.5*iqr) | (sub.Sale_Price_AUD > q3 + 1.5*iqr)]
    for _, r in flagged.iterrows():
        print(f"{r.Property_ID} ({s}): {r.Property_Type}, {r.Bedrooms} bed, ${r.Sale_Price_AUD:,.0f}")

Outliers
CB-015 (Campbelltown): House, 7 bed, $1,750,000


In [20]:
#feature correlations with price
print("\nCorrelation with Sale_Price_AUD")
num_cols = ["Bedrooms", "Bathrooms", "Parking_Spaces", "Land_Size_sqm", "Distance_to_CBD_km"]
print(df[num_cols + ["Sale_Price_AUD"]].corr()["Sale_Price_AUD"].drop("Sale_Price_AUD").round(3))

#feature engineering
df["Dwelling_Group"] = df.Property_Type.apply(lambda x: 1 if x in ["House","Villa","Semi-detached","Terrace"] else 0)
df["Total_Rooms"] = df.Bedrooms + df.Bathrooms
print("\nEngineered feature correlations")
print("Dwelling_Group (house=1):", round(df.Dwelling_Group.corr(df.Sale_Price_AUD), 3))
print("Total_Rooms:", round(df.Total_Rooms.corr(df.Sale_Price_AUD), 3))
print("\nMean price : House-like vs Unit:")
print(df.groupby(df.Dwelling_Group.map({1: "House-like", 0: "Unit"}))["Sale_Price_AUD"]
      .agg(["count", "mean"]).round(0))


Correlation with Sale_Price_AUD
Bedrooms              0.807
Bathrooms             0.388
Parking_Spaces        0.553
Land_Size_sqm         0.681
Distance_to_CBD_km    0.624
Name: Sale_Price_AUD, dtype: float64

Engineered feature correlations
Dwelling_Group (house=1): 0.686
Total_Rooms: 0.729

Mean price : House-like vs Unit:
                count       mean
Dwelling_Group                  
House-like         65  1031147.0
Unit               37   594162.0


In [ ]:
df["Dwelling_Group"] = df.Property_Type.apply(lambda x: 1 if x in ["House","Villa","Semi-detached","Terrace"] else 0)
df["Has_Land_Size"] = df.Land_Size_sqm.notna().astype(int)
house_medians = df[df.Dwelling_Group == 1].groupby("Suburb")["Land_Size_sqm"].median()

def fill_land(row):
    if pd.notna(row.Land_Size_sqm):
        return row.Land_Size_sqm
    if row.Dwelling_Group == 0:
        return 0.0
    return house_medians[row.Suburb]

df["Land_Size_sqm"] = df.apply(fill_land, axis=1)
df["Parking_Spaces"] = df["Parking_Spaces"].fillna(df["Parking_Spaces"].median())

#prepare another dataset with median estimates for filling gaps in the rows, keep this and raw data separate
df.to_csv("model_ready.csv", index=False)
print(df[["Bedrooms","Bathrooms","Parking_Spaces","Land_Size_sqm","Has_Land_Size","Dwelling_Group"]].isna().sum())

Bedrooms          0
Bathrooms         0
Parking_Spaces    0
Land_Size_sqm     0
Has_Land_Size     0
Dwelling_Group    0
dtype: int64


In [27]:
from sklearn.linear_model import LinearRegression
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import KFold, cross_validate

df = pd.read_csv("model_ready.csv")

X = pd.get_dummies(df[["Suburb"]], drop_first=True)
X = pd.concat([
    df[["Bedrooms", "Bathrooms", "Parking_Spaces", "Land_Size_sqm", "Has_Land_Size", "Dwelling_Group"]],X], axis=1)
y = df["Sale_Price_AUD"]

#three regression models
models = {
    "Linear Regression": LinearRegression(),
    "Decision Tree": DecisionTreeRegressor(random_state=42),
    "Random Forest": RandomForestRegressor(n_estimators=300, random_state=42),
}

#evaluate using k-fold cross validation
kf = KFold(n_splits=5, shuffle=True, random_state=42)
scoring = {"rmse": "neg_root_mean_squared_error", "mae": "neg_mean_absolute_error", "r2": "r2"}

for name, model in models.items():
    cv = cross_validate(model, X, y, cv=kf, scoring=scoring)
    print(name, "RMSE:", (-cv["test_rmse"]).mean(), "MAE:", (-cv["test_mae"]).mean(), "R2:", cv["test_r2"].mean())

    model.fit(X, y)
    train_r2 = model.score(X, y)
    print(f"  train R2 = {train_r2:.3f}, CV R2 = {cv['test_r2'].mean():.3f}, gap = {train_r2 - cv['test_r2'].mean():.3f}")

Linear Regression RMSE: 160417.23286339152 MAE: 116555.62119558922 R2: 0.6934347515633377
  train R2 = 0.771, CV R2 = 0.693, gap = 0.077
Decision Tree RMSE: 192490.62800716804 MAE: 136272.7780013637 R2: 0.5467137023670247
  train R2 = 0.979, CV R2 = 0.547, gap = 0.432
Random Forest RMSE: 156857.11115663458 MAE: 118952.80803848481 R2: 0.681344660309068
  train R2 = 0.946, CV R2 = 0.681, gap = 0.265
